# Session 9 — Loss Harness & Output Heads

Assignment: build one notebook, one loss harness, and make everything it does
*observable* rather than assumed. Every claim below is checked in code in the
cell that follows it — shapes are printed, targets are decoded back to
strings, masking is proven by a token count that changes, and memory numbers
come from the real allocator, not arithmetic on paper.

**Part 1** instruments `hidden = model(tokens); logits = head(hidden); loss =
cross_entropy(...)` against seven checks. **Part 2** adds a second output
head that predicts `t+2` instead of `t+1` and compares how its loss behaves
against the first head over training.

Model: a ~15M-parameter decoder-only transformer using the two building
blocks from this session's class — **RMSNorm** (pre-norm, no centering, just
projects onto a sphere) and **SwiGLU** feed-forward (`W2(silu(W1 x) * Vx)`,
the gated design, not a ReLU MLP). Tokenizer: a byte-level BPE trained from
scratch on the training corpus itself (offline, no network calls once this
notebook has the data file), vocab size ~6000 — big enough that the
tied-vs-untied head comparison in 1.6 is a real, visible parameter delta,
not a rounding error the way it would be with a ~65-token char vocab.

In [1]:
import math, os, time, json, threading
import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import ByteLevelBPETokenizer

torch.manual_seed(1337)

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

os.makedirs("artifacts", exist_ok=True)
os.makedirs("submission_artifacts", exist_ok=True)

device: mps


## Data & tokenizer

`data/tinyshakespeare.txt` is the standard Karpathy tiny-Shakespeare corpus,
committed to the repo so this notebook needs no network access to run. We
train a byte-level BPE tokenizer directly on it — this is the "token is a
subword" idea from the session: no per-language dictionary, just the merges
this specific corpus supports, plus `<pad>` and `<eos>` as reserved specials.

In [2]:
DATA_PATH = "data/tinyshakespeare.txt"
text = open(DATA_PATH, encoding="utf-8").read()
print("corpus characters:", len(text))

VOCAB_SIZE = 6000
tok = ByteLevelBPETokenizer()
tok.train([DATA_PATH], vocab_size=VOCAB_SIZE, min_frequency=2,
          special_tokens=["<pad>", "<eos>"])
tok.save_model("artifacts", "tokenizer")

vocab_size = tok.get_vocab_size()
pad_id = tok.token_to_id("<pad>")
eos_id = tok.token_to_id("<eos>")
print("vocab_size:", vocab_size, " pad_id:", pad_id, " eos_id:", eos_id)

ids = tok.encode(text).ids
print("total tokens:", len(ids))
print("chars per token (compression):", round(len(text) / len(ids), 3))

data = torch.tensor(ids, dtype=torch.long)
n_split = int(0.9 * len(data))
train_data, val_data = data[:n_split], data[n_split:]
print("train tokens:", len(train_data), " val tokens:", len(val_data))

def get_batch(split, batch_size, block_size):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + 1 + block_size] for i in ix])
    return x.to(device), y.to(device)

corpus characters: 1115394





vocab_size: 6000  pad_id: 0  eos_id: 1


total tokens: 327608
chars per token (compression): 3.405
train tokens: 294847  val tokens: 32761


## Model: RMSNorm + SwiGLU decoder

- `RMSNorm` — no mean-centering, scales activations onto a fixed-radius
  sphere: `x * rsqrt(mean(x**2) + eps) * weight`.
- `SwiGLU` — the gated FFN design from the session: two input projections
  (`W1`, `V`), `silu` gate on one, elementwise product, then `W2` back down.
  Hidden width uses the `8/3` multiplier (rounded to a multiple of 8) that
  keeps SwiGLU's parameter count comparable to a plain 4x ReLU MLP, since
  SwiGLU has three weight matrices instead of two.
- Causal self-attention, pre-norm residual stream (`x = x + block(norm(x))`
  — the residual line itself is never transformed, only added to).
- `TinyGPT.forward` returns `(hidden, logits, logits2)` — `logits2` is
  `None` unless `extra_head_offset` is set, which is Part 2's second head.

In [3]:
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))

    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return x * rms * self.weight


class SwiGLU(nn.Module):
    def __init__(self, d, mult=8 / 3):
        super().__init__()
        hidden = int(d * mult)
        hidden = ((hidden + 7) // 8) * 8
        self.w1 = nn.Linear(d, hidden, bias=False)
        self.v = nn.Linear(d, hidden, bias=False)
        self.w2 = nn.Linear(hidden, d, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.v(x))


class CausalSelfAttention(nn.Module):
    def __init__(self, d, n_head, block_size):
        super().__init__()
        assert d % n_head == 0
        self.n_head = n_head
        self.d_head = d // n_head
        self.qkv = nn.Linear(d, 3 * d, bias=False)
        self.proj = nn.Linear(d, d, bias=False)
        mask = torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size)
        self.register_buffer("mask", mask)

    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).split(D, dim=2)
        q = q.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, D)
        return self.proj(y)


class Block(nn.Module):
    def __init__(self, d, n_head, block_size):
        super().__init__()
        self.norm1 = RMSNorm(d)
        self.attn = CausalSelfAttention(d, n_head, block_size)
        self.norm2 = RMSNorm(d)
        self.ffn = SwiGLU(d)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


class TinyGPT(nn.Module):
    def __init__(self, vocab_size, d_model=384, n_layer=6, n_head=6, block_size=1024,
                 tie_weights=True, extra_head_offset=None):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.blocks = nn.ModuleList([Block(d_model, n_head, block_size) for _ in range(n_layer)])
        self.norm_f = RMSNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.tie_weights = tie_weights
        if tie_weights:
            self.head.weight = self.tok_emb.weight
        self.extra_head_offset = extra_head_offset
        if extra_head_offset is not None:
            self.head2 = nn.Linear(d_model, vocab_size, bias=False)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def hidden(self, tokens):
        B, T = tokens.shape
        pos = torch.arange(T, device=tokens.device)
        x = self.tok_emb(tokens) + self.pos_emb(pos)[None, :, :]
        for blk in self.blocks:
            x = blk(x)
        return self.norm_f(x)

    def forward(self, tokens):
        h = self.hidden(tokens)
        logits = self.head(h)
        logits2 = self.head2(h) if self.extra_head_offset is not None else None
        return h, logits, logits2


def count_params(model, include_head2=False):
    total, seen = 0, set()
    for name, p in model.named_parameters():
        if not include_head2 and name.startswith("head2"):
            continue
        if id(p) in seen:
            continue
        seen.add(id(p))
        total += p.numel()
    return total


BLOCK_SIZE = 1024
D_MODEL = 384
N_LAYER = 6
N_HEAD = 6

model = TinyGPT(vocab_size, D_MODEL, N_LAYER, N_HEAD, BLOCK_SIZE,
                 tie_weights=False, extra_head_offset=2).to(device)
print("model params (head untied, incl. unused head2 buffer for shape demo):",
      count_params(model, include_head2=True))

model params (head untied, incl. unused head2 buffer for shape demo): 17927040


## Part 1.1 — every tensor shape, one line each

`tokens` is `[B, T]`. `hidden` is `[B, T, D]` — the trunk's job is done here,
it has thought about every position but hasn't scored a single vocabulary
word yet. `logits` is `[B, T, V]` — this is the tensor Widget 7 calls out as
"32x the hidden state": going from `D` to `V` per position is the expensive
step, not the trunk.

In [4]:
B, T = 4, 32
tokens, targets = get_batch("train", B, T)
h, logits, logits2 = model(tokens)

print(f"tokens   shape = {str(tuple(tokens.shape)):<18} (B={B}, T={T})")
print(f"targets  shape = {str(tuple(targets.shape)):<18} (B={B}, T={T})")
print(f"hidden   shape = {str(tuple(h.shape)):<18} (B={B}, T={T}, D={D_MODEL})")
print(f"logits   shape = {str(tuple(logits.shape)):<18} (B={B}, T={T}, V={vocab_size})")
print(f"logits2  shape = {str(tuple(logits2.shape)):<18} (B={B}, T={T}, V={vocab_size})  [Part 2's t+2 head]")
print(f"logits / hidden element-count ratio: {logits.numel() / h.numel():.1f}x  (== V/D = {vocab_size / D_MODEL:.1f})")

tokens   shape = (4, 32)            (B=4, T=32)
targets  shape = (4, 32)            (B=4, T=32)
hidden   shape = (4, 32, 384)       (B=4, T=32, D=384)
logits   shape = (4, 32, 6000)      (B=4, T=32, V=6000)
logits2  shape = (4, 32, 6000)      (B=4, T=32, V=6000)  [Part 2's t+2 head]
logits / hidden element-count ratio: 15.6x  (== V/D = 15.6)


## Part 1.2 — verify the shift with strings, not ids

The instructor's warning: a target shift in the wrong direction can still
produce a plausible-looking loss scalar on an untrained model — it will not
raise an exception. The only way to catch it is to decode `input[i]` and
`target[i]` back to text and check by eye that `target[i] == input[i+1]`.
Below, `correct_shift_loss` uses the real `targets = tokens shifted by one`;
`wrong_shift_loss` is a deliberate foil where the "target" is the input
token itself (a same-position, i.e. zero-shift, bug) — printed side by side
to show neither loss value alone tells you which one is correct.

In [5]:
row, tgt = tokens[0].tolist(), targets[0].tolist()
print(f"{'pos':>4}  {'input':<14} {'target':<14} next-token-of-input (should equal target)")
for i in range(8):
    nxt = tok.decode([row[i + 1]]) if i + 1 < len(row) else "<end>"
    print(f"{i:>4}  {tok.decode([row[i]])!r:<14} {tok.decode([tgt[i]])!r:<14} {nxt!r}")

correct_shift_loss = F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1)).item()
wrong_shift_targets = tokens.clone()          # bug: target == input (no shift at all)
wrong_shift_loss = F.cross_entropy(logits.view(-1, vocab_size), wrong_shift_targets.view(-1)).item()

print(f"\nloss, CORRECT shift (target = next token): {correct_shift_loss:.4f}")
print(f"loss, WRONG shift   (target = same token):  {wrong_shift_loss:.4f}")
print("Both numbers look like ordinary untrained-model losses -- the scalar alone cannot")
print("tell you the shift is wrong. Only the decoded strings above can.")

 pos  input          target         next-token-of-input (should equal target)
   0  ' back'        ' of'          ' of'
   1  ' of'          ' Montague'    ' Montague'
   2  ' Montague'    ',--'          ',--'
   3  ',--'          '\n'           '\n'
   4  '\n'           'And'          'And'
   5  'And'          ' it'          ' it'
   6  ' it'          ' mis'         ' mis'
   7  ' mis'         '-'            '-'

loss, CORRECT shift (target = next token): 8.7695
loss, WRONG shift   (target = same token):  8.7499
Both numbers look like ordinary untrained-model losses -- the scalar alone cannot
tell you the shift is wrong. Only the decoded strings above can.


## Part 1.3 — mask padding, confirm the contributing-token count changes

Build a padded batch (two short sequences in a 12-token frame) and compute
the loss with and without `ignore_index=pad_id`. Predicting `<pad>` is
trivial for the model, so leaving it unmasked would let loss look better
than the model's real, on-content performance without the model having
learned anything about content. The number that must change here is the
count of tokens contributing to the mean.

In [6]:
Bp, Tp = 2, 12
real_len = [9, 5]
pad_tokens = torch.full((Bp, Tp), pad_id, dtype=torch.long)
pad_targets = torch.full((Bp, Tp), pad_id, dtype=torch.long)
for b, L in enumerate(real_len):
    body = torch.randint(2, vocab_size, (L,))
    pad_tokens[b, :L] = body
    pad_targets[b, :L - 1] = body[1:]
    pad_targets[b, L - 1] = eos_id
pad_tokens, pad_targets = pad_tokens.to(device), pad_targets.to(device)

_, plog, _ = model(pad_tokens)
plog_flat, tgt_flat = plog.view(-1, vocab_size), pad_targets.view(-1)

loss_unmasked = F.cross_entropy(plog_flat, tgt_flat)
n_unmasked = tgt_flat.numel()
loss_masked = F.cross_entropy(plog_flat, tgt_flat, ignore_index=pad_id)
n_masked = int((tgt_flat != pad_id).sum().item())

print(f"positions total:                       {tgt_flat.numel()}")
print(f"contributing tokens, WITHOUT pad mask:  {n_unmasked}   loss={loss_unmasked.item():.4f}")
print(f"contributing tokens, WITH    pad mask:  {n_masked}   loss={loss_masked.item():.4f}")
assert n_masked < n_unmasked
print(f"count changed as required: {n_masked} < {n_unmasked}")

positions total:                       24
contributing tokens, WITHOUT pad mask:  24   loss=8.5438
contributing tokens, WITH    pad mask:  14   loss=8.9236
count changed as required: 14 < 24


## Part 1.4 — pack two documents, mask the boundary

Concatenate two unrelated short passages with `<eos>` as the separator. The
`<eos>` position's naive "next token" target is the *first token of the
other document* — a document-boundary leak. We show the mean loss before
and after excluding that one boundary position from the loss.

In [7]:
doc_a = tok.encode("First Citizen:\nBefore we proceed any further, hear me speak.").ids
doc_b = tok.encode("Second Citizen:\nWhat you have said sounds like treason to me.").ids
packed = torch.tensor(doc_a + [eos_id] + doc_b, dtype=torch.long).unsqueeze(0)

pk_in = packed[:, :-1].to(device)
pk_tgt_raw = packed[:, 1:].clone()
boundary_pos = len(doc_a)
assert pk_in[0, boundary_pos].item() == eos_id

leaked_token = tok.decode([pk_tgt_raw[0, boundary_pos].item()])
print(f"<eos> sits at position {boundary_pos}; its naive target is {leaked_token!r} "
      f"-- the first token of the unrelated doc B")

pk_tgt_masked = pk_tgt_raw.clone()
pk_tgt_masked[0, boundary_pos] = pad_id  # reuse pad_id as this loss mask's ignore_index
pk_tgt_raw, pk_tgt_masked = pk_tgt_raw.to(device), pk_tgt_masked.to(device)

_, pk_logits, _ = model(pk_in)
loss_before = F.cross_entropy(pk_logits.view(-1, vocab_size), pk_tgt_raw.view(-1)).item()
loss_after = F.cross_entropy(pk_logits.view(-1, vocab_size), pk_tgt_masked.view(-1), ignore_index=pad_id).item()

print(f"loss BEFORE masking the boundary: {loss_before:.4f}")
print(f"loss AFTER  masking the boundary: {loss_after:.4f}")
print(f"delta: {loss_before - loss_after:+.4f}")
print("The delta is that one boundary position's own NLL term leaving the mean -- with two")
print("independent documents packed together, the model has no business being scored on")
print("predicting doc B's opening token from doc A's <eos>.")

<eos> sits at position 14; its naive target is 'Second' -- the first token of the unrelated doc B
loss BEFORE masking the boundary: 8.7375
loss AFTER  masking the boundary: 8.7532
delta: -0.0157
The delta is that one boundary position's own NLL term leaving the mean -- with two
independent documents packed together, the model has no business being scored on
predicting doc B's opening token from doc A's <eos>.


## Part 1.5 — perplexity of an untrained model

At random initialization the model has no information, so its predictive
distribution over the vocabulary should be close to uniform. The
information-theoretic floor for a uniform guess over `V` tokens is
`-log(1/V) = ln(V)` nats, i.e. perplexity `== V`. If a fresh model's loss
does not start near `ln(V)`, something is wrong before any training begins
(data leakage, a bad target shift, wrong loss reduction, etc.) — per the
instructor: *"if the loss does not start between 11 and 12 [for V=131072],
something is wrong, do not train."* Same rule, our `V`.

In [8]:
fresh = TinyGPT(vocab_size, D_MODEL, N_LAYER, N_HEAD, BLOCK_SIZE, tie_weights=False).to(device)
fresh.eval()
with torch.no_grad():
    xb, yb = get_batch("val", 8, 256)
    _, flog, _ = fresh(xb)
    fresh_loss = F.cross_entropy(flog.view(-1, vocab_size), yb.view(-1)).item()

fresh_ppl = math.exp(fresh_loss)
theoretical_loss = math.log(vocab_size)

print(f"measured loss (untrained):       {fresh_loss:.4f} nats")
print(f"measured perplexity (untrained): {fresh_ppl:.1f}")
print(f"theoretical ln(V):               {theoretical_loss:.4f} nats   (V={vocab_size})")
print(f"theoretical perplexity == V:     {vocab_size}")
print(f"measured / theoretical loss ratio: {fresh_loss / theoretical_loss:.4f}  (should be close to 1.0)")
del fresh

measured loss (untrained):       8.7500 nats
measured perplexity (untrained): 6310.4
theoretical ln(V):               8.6995 nats   (V=6000)
theoretical perplexity == V:     6000
measured / theoretical loss ratio: 1.0058  (should be close to 1.0)


## Part 1.6 — tied vs. untied output head, parameter count

Tying reuses the token embedding matrix (`[V, D]`) as the output head, so
the only saving is exactly one `[V, D]` matrix's worth of parameters — the
same idea from Session 7, just at a scale where it's still large relative to
the whole model (unlike a char-level ~65-token vocab, where this saving
would round to noise).

In [9]:
m_tied = TinyGPT(vocab_size, D_MODEL, N_LAYER, N_HEAD, BLOCK_SIZE, tie_weights=True)
m_untied = TinyGPT(vocab_size, D_MODEL, N_LAYER, N_HEAD, BLOCK_SIZE, tie_weights=False)
p_tied = count_params(m_tied)
p_untied = count_params(m_untied)

print(f"tied head params:   {p_tied:,}")
print(f"untied head params: {p_untied:,}")
print(f"difference:         {p_untied - p_tied:,}   (== vocab_size * d_model = {vocab_size * D_MODEL:,})")
print(f"saving as a fraction of the untied total: {(p_untied - p_tied) / p_untied:.1%}")
del m_tied, m_untied

tied head params:   13,319,040
untied head params: 15,623,040
difference:         2,304,000   (== vocab_size * d_model = 2,304,000)
saving as a fraction of the untied total: 14.7%


## Part 1.7 — peak memory: ordinary cross-entropy vs. a chunked version

Widget 7's point: the logits tensor `[N, V]` (`N` = tokens in the batch) is
the expensive object in this whole computation, far bigger than the hidden
state that produced it. Widget 8's point: you can compute the *exact same*
mean loss by looping over chunks of `N`, materializing one small
`[chunk, V]` logits tensor at a time instead of one giant one — at the cost
of extra passes through the output head, in exchange for peak memory that
scales with `chunk_size` instead of `N`.

We isolate the loss module itself here: a synthetic `hidden` tensor of
realistic size stands in for "the hidden state that came out of the trunk"
(the trunk's own memory footprint is a separate, already-demonstrated
concern — see 1.1), and the *real* output head (same `vocab_size` as the
trained model) is what's under test. `PeakMemory` reads the real allocator:
`torch.cuda.max_memory_allocated()` on CUDA, a polling thread against
`torch.mps.current_allocated_memory()` on Apple Silicon (MPS has no native
peak counter), or `tracemalloc` as a CPU fallback.

In [10]:
class PeakMemory:
    def __init__(self, device, interval=0.0005):
        self.device = device
        self.interval = interval
        self.peak = 0

    def _poll(self):
        while not self._stop:
            cur = torch.mps.current_allocated_memory()
            if cur > self.peak:
                self.peak = cur
            time.sleep(self.interval)

    def __enter__(self):
        if self.device == "cuda":
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
        elif self.device == "mps":
            torch.mps.synchronize()
            torch.mps.empty_cache()
            self.peak = torch.mps.current_allocated_memory()
            self._stop = False
            self._thread = threading.Thread(target=self._poll, daemon=True)
            self._thread.start()
        else:
            import tracemalloc
            tracemalloc.start()
        return self

    def __exit__(self, *a):
        if self.device == "cuda":
            torch.cuda.synchronize()
            self.peak = torch.cuda.max_memory_allocated()
        elif self.device == "mps":
            torch.mps.synchronize()
            self._stop = True
            self._thread.join()
        else:
            import tracemalloc
            _, self.peak = tracemalloc.get_traced_memory()
            tracemalloc.stop()


def ordinary_ce(head, hidden_leaf, targets_flat, ignore_index):
    logits = head(hidden_leaf)
    loss = F.cross_entropy(logits, targets_flat, ignore_index=ignore_index)
    loss.backward()
    return loss.detach()


def chunked_ce(head, hidden_leaf, targets_flat, ignore_index, chunk_size):
    total_count = (targets_flat != ignore_index).sum().clamp(min=1)
    total_loss = torch.zeros((), device=hidden_leaf.device)
    N = hidden_leaf.shape[0]
    for start in range(0, N, chunk_size):
        end = min(start + chunk_size, N)
        h_chunk, t_chunk = hidden_leaf[start:end], targets_flat[start:end]
        logits_chunk = head(h_chunk)
        loss_chunk = F.cross_entropy(logits_chunk, t_chunk, ignore_index=ignore_index,
                                      reduction="sum") / total_count
        loss_chunk.backward()
        total_loss += loss_chunk.detach()
        del logits_chunk, loss_chunk
    return total_loss


mem_head = nn.Linear(D_MODEL, vocab_size, bias=False).to(device)
Bm, Tm, Nm, CHUNK = 16, 2048, 16 * 2048, 1024
torch.manual_seed(0)
hidden_src = torch.randn(Nm, D_MODEL, device=device) * 0.02
targets_mem = torch.randint(2, vocab_size, (Nm,), device=device)
targets_mem[::37] = pad_id

mem_head.zero_grad(set_to_none=True)
h1 = hidden_src.clone().requires_grad_(True)
with PeakMemory(device) as pm_naive:
    loss_naive = ordinary_ce(mem_head, h1, targets_mem, pad_id)
del h1
if device == "mps":
    torch.mps.empty_cache()

mem_head.zero_grad(set_to_none=True)
h2 = hidden_src.clone().requires_grad_(True)
with PeakMemory(device) as pm_chunk:
    loss_chunk = chunked_ce(mem_head, h2, targets_mem, pad_id, CHUNK)
del h2
if device == "mps":
    torch.mps.empty_cache()

print(f"tokens N={Nm}, vocab V={vocab_size}, chunk={CHUNK} ({Nm // CHUNK} chunks)")
print(f"loss (ordinary, one pass): {loss_naive.item():.10f}")
print(f"loss (chunked):            {loss_chunk.item():.10f}")
print(f"|difference|:              {abs(loss_naive.item() - loss_chunk.item()):.2e}  (same objective)")
print(f"peak memory, ordinary: {pm_naive.peak / 1e6:9.2f} MB")
print(f"peak memory, chunked:  {pm_chunk.peak / 1e6:9.2f} MB")
print(f"ratio (ordinary / chunked): {pm_naive.peak / pm_chunk.peak:.2f}x")

tokens N=32768, vocab V=6000, chunk=1024 (32 chunks)
loss (ordinary, one pass): 8.6995220184
loss (chunked):            8.6995220184
|difference|:              0.00e+00  (same objective)
peak memory, ordinary:   3451.70 MB
peak memory, chunked:     480.39 MB
ratio (ordinary / chunked): 7.19x


## Part 1 — the seven numbers, consolidated

Reading the checklist as seven concrete, quantitative deliverables:

In [11]:
part1_numbers = {
    "1_padding_contributing_tokens": {"with_mask": n_masked, "without_mask": n_unmasked},
    "2_doc_boundary_loss_delta": loss_before - loss_after,
    "3_untrained_perplexity": fresh_ppl,
    "4_tied_head_params": p_tied,
    "5_untied_head_params": p_untied,
    "6_peak_memory_ordinary_MB": pm_naive.peak / 1e6,
    "7_peak_memory_chunked_MB": pm_chunk.peak / 1e6,
    "peak_memory_ratio": pm_naive.peak / pm_chunk.peak,
    "device_used": device,
}
for k, v in part1_numbers.items():
    print(f"{k:35s} {v}")

with open("submission_artifacts/part1_numbers.json", "w") as f:
    json.dump(part1_numbers, f, indent=2)

1_padding_contributing_tokens       {'with_mask': 14, 'without_mask': 24}
2_doc_boundary_loss_delta           -0.015720367431640625
3_untrained_perplexity              6310.375162108038
4_tied_head_params                  13319040
5_untied_head_params                15623040
6_peak_memory_ordinary_MB           3451.695616
7_peak_memory_chunked_MB            480.392192
peak_memory_ratio                   7.185161777150616
device_used                         mps


---
# Part 2 — a second head predicting `t+2`

One trunk, two output heads reading the same hidden state `h_t`: `head`
predicts `t+1` as before, `head2` predicts `t+2`. In training both heads
always see real tokens as context (never each other's predictions — that
speculative-decoding trick from the session's MTP discussion is
inference-only); both losses are computed against the real next-next token
and summed for backprop, exactly as `L = L1 + L2` in Widget 5.

In [12]:
def get_batch_mtp(split, batch_size, block_size):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size - 2, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y1 = torch.stack([d[i + 1:i + 1 + block_size] for i in ix])
    y2 = torch.stack([d[i + 2:i + 2 + block_size] for i in ix])
    return x.to(device), y1.to(device), y2.to(device)


MTP_BLOCK = 256
mtp_model = TinyGPT(vocab_size, D_MODEL, N_LAYER, N_HEAD, BLOCK_SIZE,
                     tie_weights=True, extra_head_offset=2).to(device)
n_params_mtp = count_params(mtp_model, include_head2=True)
n_params_head2 = mtp_model.head2.weight.numel()
print(f"MTP model params (incl. head2): {n_params_mtp:,}   (head2 alone: {n_params_head2:,})")

opt = torch.optim.AdamW(mtp_model.parameters(), lr=3e-4, weight_decay=0.01)
STEPS, EVAL_EVERY, EVAL_BATCHES = 400, 20, 5

@torch.no_grad()
def eval_losses():
    mtp_model.eval()
    l1s, l2s = [], []
    for _ in range(EVAL_BATCHES):
        xb, y1b, y2b = get_batch_mtp("val", 16, MTP_BLOCK)
        _, lg1, lg2 = mtp_model(xb)
        l1s.append(F.cross_entropy(lg1.reshape(-1, vocab_size), y1b.reshape(-1)).item())
        l2s.append(F.cross_entropy(lg2.reshape(-1, vocab_size), y2b.reshape(-1)).item())
    mtp_model.train()
    return sum(l1s) / len(l1s), sum(l2s) / len(l2s)

history = []
t0 = time.time()
for step in range(1, STEPS + 1):
    xb, y1b, y2b = get_batch_mtp("train", 32, MTP_BLOCK)
    _, lg1, lg2 = mtp_model(xb)
    loss1 = F.cross_entropy(lg1.reshape(-1, vocab_size), y1b.reshape(-1))
    loss2 = F.cross_entropy(lg2.reshape(-1, vocab_size), y2b.reshape(-1))
    loss = loss1 + loss2
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    if step % EVAL_EVERY == 0 or step == 1:
        el1, el2 = eval_losses()
        history.append((step, el1, el2))
        print(f"step {step:4d}  val L1={el1:.4f}  val L2={el2:.4f}  sum={el1 + el2:.4f}  gap(L2-L1)={el2 - el1:+.4f}")

print(f"training time: {time.time() - t0:.1f}s on {device}")

MTP model params (incl. head2): 15,623,040   (head2 alone: 2,304,000)


step    1  val L1=8.3386  val L2=8.3881  sum=16.7267  gap(L2-L1)=+0.0495


step   20  val L1=6.6379  val L2=6.5793  sum=13.2173  gap(L2-L1)=-0.0586


step   40  val L1=6.0205  val L2=6.1581  sum=12.1786  gap(L2-L1)=+0.1376


step   60  val L1=5.8818  val L2=6.0638  sum=11.9456  gap(L2-L1)=+0.1821


...

step  320  val L1=4.9504  val L2=5.6548  sum=10.6053  gap(L2-L1)=+0.7044


step  340  val L1=4.9109  val L2=5.6136  sum=10.5245  gap(L2-L1)=+0.7028


step  360  val L1=4.9449  val L2=5.6809  sum=10.6258  gap(L2-L1)=+0.7360


step  380  val L1=4.8916  val L2=5.6002  sum=10.4918  gap(L2-L1)=+0.7086


step  400  val L1=4.7478  val L2=5.4469  sum=10.1948  gap(L2-L1)=+0.6991
training time: 189.7s on mps


## Part 2 — plot and the two losses

In [13]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

steps = [s for s, _, _ in history]
l1s = [a for _, a, _ in history]
l2s = [b for _, _, b in history]

plt.figure(figsize=(7, 4.5))
plt.plot(steps, l1s, label="L1 (predict t+1)", linewidth=2)
plt.plot(steps, l2s, label="L2 (predict t+2)", linewidth=2)
plt.xlabel("training step")
plt.ylabel("val loss (nats)")
plt.title("Two heads, one trunk: L1 vs L2 over training")
plt.legend()
plt.tight_layout()
plt.savefig("submission_artifacts/mtp_losses.png", dpi=150)
plt.show()

final_l1, final_l2 = history[-1][1], history[-1][2]
first_l1, first_l2 = history[0][1], history[0][2]

print(f"FIRST eval  L1={first_l1:.4f}  L2={first_l2:.4f}  gap={first_l2 - first_l1:+.4f}")
print(f"FINAL eval  L1={final_l1:.4f}  L2={final_l2:.4f}  sum={final_l1 + final_l2:.4f}  gap={final_l2 - final_l1:+.4f}")
print(f"L1 improvement over training: {first_l1 - final_l1:.4f} nats")
print(f"L2 improvement over training: {first_l2 - final_l2:.4f} nats")

part2_losses = {
    "steps_trained": STEPS,
    "L1_final": final_l1,
    "L2_final": final_l2,
    "sum_final": final_l1 + final_l2,
    "L1_first": first_l1,
    "L2_first": first_l2,
    "gap_first": first_l2 - first_l1,
    "gap_final": final_l2 - final_l1,
    "L1_improvement_nats": first_l1 - final_l1,
    "L2_improvement_nats": first_l2 - final_l2,
    "history": history,
}
with open("submission_artifacts/part2_losses.json", "w") as f:
    json.dump(part2_losses, f, indent=2)

FIRST eval  L1=8.3386  L2=8.3881  gap=+0.0495
FINAL eval  L1=4.7478  L2=5.4469  sum=10.1948  gap=+0.6991
L1 improvement over training: 3.5908 nats
L2 improvement over training: 2.9412 nats


/var/folders/8y/jyfglb6x63qc3q3l42r445r80000gn/T/ipykernel_33013/2919115623.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Part 2 — what happens to the second head's loss, and why

`L2` (predicting `t+2`) starts essentially tied with `L1` (predicting
`t+1`) — at initialization neither head has learned anything, so the gap is
noise. As training progresses, `L1` drops faster and `L2` trails behind it,
and the gap between them *widens* rather than closing. See
`gap_first` vs `gap_final` printed above.

This matches the session's Widget 5 finding (corpus loss climbing from
head 1 to head 4 as prediction distance increases) and the mechanical
reason given in class: both heads read off the *same* hidden state `h_t` —
neither one gets to condition on the other's prediction during training
(only real tokens are fed in, one at a time). `head` only has to answer
"what comes right after this context" — the hardest part of that answer is
already implicit in `h_t`. `head2` has to answer a strictly harder question
from the *same* information: "what comes after that, without knowing what
comes right after this" — it has to implicitly marginalize over the unknown
`t+1` using only `h_t`, which is strictly less information than `head`
effectively gets to use. The extra distance costs nats, and the gap is the
direct, measured price of that missing intermediate token.